# [WIP] Proposed Redesign of SB-MOABB DataIO


In [1]:
%load_ext line_profiler
%load_ext memory_profiler

In [1]:
%%capture
!pip install speechbrain moabb mne mne_bids braindecode

# Extend DynamicItemDataset to support BIDS and MOABB datasets


In [2]:
import mne
import moabb
import numpy as np
from moabb.datasets import BNCI2014_001

mne.set_log_level(verbose=False)
moabb.set_log_level(level="ERROR")

In [3]:
from dataio.datasets import RawEEGDataset, EpochedEEGDataset

In [6]:
%%memit
dataset = EpochedEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001-epoched.json",
    save_path="data",
    tmin=0,
    tmax=4.0,
    output_keys=[
        "label",
        "subject",
        "session",
        "epoch",
    ],
)

for _ in dataset:
    pass


peak memory: 779.36 MiB, increment: 42.89 MiB


In [7]:
%%time
for _ in dataset:
    pass

CPU times: user 1.71 s, sys: 98.6 ms, total: 1.8 s
Wall time: 1.8 s


In [8]:
%%memit
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import create_windows_from_events

raw_braindecode_dataset = MOABBDataset("BNCI2014_001", subject_ids=None)
epoched_braindecode_dataset = create_windows_from_events(raw_braindecode_dataset)

for sample in epoched_braindecode_dataset:
    pass

peak memory: 2928.79 MiB, increment: 2149.43 MiB


In [9]:
%%timeit

for sample in epoched_braindecode_dataset:
    pass

155 ms ± 1.52 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [10]:
%%memit
dataset = EpochedEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001-epoched.json",
    save_path="data",
    tmin=0,
    tmax=4.0,
    output_keys=[
        "label",
        "subject",
        "session",
        "epoch",
    ],
    preload=True
)

for _ in dataset:
    pass


peak memory: 4623.27 MiB, increment: 1779.12 MiB


In [11]:
%%timeit
for _ in dataset:
    pass

154 ms ± 4.21 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
print(dataset[0]["epoch"].shape, epoched_braindecode_dataset[0][0].shape)
np.allclose(dataset[0]["epoch"], epoched_braindecode_dataset[0][0][:22])

(22, 1000) (26, 1000)


True

In [18]:
import os
import torch
import torchaudio
from speechbrain.utils.data_pipeline import takes, provides
from functools import cache
import scipy.signal
import mne


hparams = dict(target_sampling_frequency=125, fmin=0.1, fmax=22)

cached_firwin = cache(scipy.signal.firwin)


@takes("epoch")
@provides("epoch")
def to_tensor(epoch):
    return torch.from_numpy(epoch).float()


@takes("epoch", "info")
@provides("epoch")
def bandpass_resample(epoch, info):
    # TODO: Align this with mne.filter.create_filter "auto" option
    n = int(round(info["sfreq"] * 0.1))
    if n % 2 == 0:
        n += 1

    bandpass = cached_firwin(
        n,
        (hparams["fmin"], hparams["fmax"]),
        fs=info["sfreq"],
        pass_zero="bandpass",
    )

    return scipy.signal.resample_poly(
        epoch,
        up=hparams["target_sampling_frequency"],
        down=info["sfreq"],
        axis=-1,
        window=bandpass,
    )


dataset = EpochedEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001-epoched.json",
    save_path="data",
    tmin=0,
    tmax=4.0,
    output_keys=[
        "label",
        "subject",
        "session",
        "epoch",
    ],
    dynamic_items=[bandpass_resample, to_tensor],
    preload=True,
)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=64, num_workers=os.cpu_count(), persistent_workers=True
)
iter(dataloader)  # create workers

In [19]:
%%timeit
for _ in dataloader:
    pass

209 ms ± 6.78 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
